In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. SCD2 pricing dimension: valid_to = next version's valid_from
p = spark.table("dbw_agentops.bronze.model_pricing")
w = Window.partitionBy("model").orderBy("valid_from")
dim = (p.withColumn("valid_from", F.to_date("effective_from"))
        .withColumn("valid_to", F.lead("valid_from").over(w))
        .select("model",
                F.col("input_usd_per_1m_tokens").cast("decimal(10,4)").alias("input_usd_per_1m"),
                F.col("output_usd_per_1m_tokens").cast("decimal(10,4)").alias("output_usd_per_1m"),
                "valid_from", "valid_to"))
dim.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dbw_agentops.gold.dim_model_pricing")

# 2. Fact table: point-in-time join to the price valid on the event date
spark.sql("""
    CREATE OR REPLACE TABLE dbw_agentops.gold.fact_agent_calls AS
    SELECT t.trace_id, t.event_time, t.event_date, t.agent_name, t.user_id, t.model,
           t.query_type, t.prompt_tokens, t.completion_tokens, t.total_tokens,
           t.tool_calls, t.latency_ms, t.status, t.error_type,
           CAST((t.prompt_tokens * d.input_usd_per_1m
               + t.completion_tokens * d.output_usd_per_1m) / 1000000 AS DECIMAL(18,8)) AS cost_usd
    FROM dbw_agentops.silver.agent_traces t
    JOIN dbw_agentops.gold.dim_model_pricing d
      ON t.model = d.model
     AND t.event_date >= d.valid_from
     AND (d.valid_to IS NULL OR t.event_date < d.valid_to)
""")

# 3. Daily metrics per agent and model
spark.sql("""
    CREATE OR REPLACE TABLE dbw_agentops.gold.agent_daily_metrics AS
    SELECT event_date, agent_name, model,
           COUNT(*) AS calls,
           SUM(total_tokens) AS total_tokens,
           CAST(SUM(cost_usd) AS DECIMAL(18,6)) AS cost_usd,
           ROUND(AVG(latency_ms), 0) AS avg_latency_ms,
           ROUND(percentile_approx(latency_ms, 0.95), 0) AS p95_latency_ms,
           ROUND(AVG(CASE WHEN status <> 'success' THEN 1.0 ELSE 0.0 END), 4) AS error_rate
    FROM dbw_agentops.gold.fact_agent_calls
    GROUP BY event_date, agent_name, model
""")

# Reconciliation: every silver row must land in the fact table with a cost
print("silver:     ", spark.table("dbw_agentops.silver.agent_traces").count())
print("fact:       ", spark.table("dbw_agentops.gold.fact_agent_calls").count())
print("null costs: ", spark.table("dbw_agentops.gold.fact_agent_calls").filter("cost_usd IS NULL").count())